# UCU - Proyecto Webscraping
Proyecto para la Licenciatura en Datos y Negocios de la Universidad Católica del Uruguay

In [10]:
import json
import re
import time
from bs4 import BeautifulSoup
import requests

CITIES_CONFIG = [
    {
        "nombre": "Montevideo",
        "url": "https://www.infocasas.com.uy/venta/casas/montevideo",
    },
    {
        "nombre": "Punta del Este",
        "url": "https://www.infocasas.com.uy/venta/casas/maldonado/punta-del-este",
    },
    {
        "nombre": "Ciudad de la Costa",
        "url": "https://www.infocasas.com.uy/venta/casas/canelones/ciudad-de-la-costa",
    },
]


def clean_number(text: str) -> int | None:

    if not text:
        return None
    numbers = re.findall(r"\d+", text.replace(".", "").replace(",", ""))
    return int(numbers[0]) if numbers else None


def scrape_city_properties(
    city_name: str, city_url: str, limit: int = 10
) -> list[dict]:

    properties = []

    try:
        response = requests.get(city_url, headers=HEADERS, timeout=10)
        response.raise_for_status()
    except requests.RequestException as e:
        print(f"Error al conectar con {city_name} ({city_url}): {e}")
        return properties

    soup = BeautifulSoup(response.content, "html.parser")


    cards = soup.select(".lc-dataCard") or soup.find_all(
        "div", class_=re.compile("property|card|listing", re.I)
    )

    for card in cards:
        if len(properties) >= limit:
            break

        try:
            # 1. Enlace de la propiedad
            link_tag = card.find("a", href=True)
            if not link_tag:
                continue
            href = link_tag["href"]
            link = (
                href
                if href.startswith("http")
                else f"https://www.infocasas.com.uy{href}"
            )

            # 2. Precio
            price_tag = card.select_one(".lc-price, .price, [class*='price']")
            price_val = (
                clean_number(price_tag.get_text()) if price_tag else None
            )

            # 3. Tamaño (m²)
            size_tag = card.find(text=re.compile(r"m²|m2", re.I))
            size_val = (
                clean_number(size_tag.parent.get_text()) if size_tag else None
            )

            # 4. Habitaciones / Dormitorios
            beds_tag = card.find(text=re.compile(r"dorm|hab|dormitorio", re.I))
            beds_val = (
                clean_number(beds_tag.parent.get_text()) if beds_tag else None
            )

            properties.append(
                {
                    "precio": price_val,
                    "tamano": size_val,
                    "habitaciones": beds_val,
                    "link": link,
                }
            )

        except Exception as e:
            print(f"Error al procesar tarjeta en {city_name}: {e}")
            continue

    print(
        f"✓ {city_name}: {len(properties)} propiedades extraídas correctamente."
    )
    return properties


def main():
    dataset = {"ciudades": []}

    for config in CITIES_CONFIG:
        print(f"Scrapeando {config['nombre']}...")
        props = scrape_city_properties(config["nombre"], config["url"], limit=10)

        dataset["ciudades"].append(
            {"nombre": config["nombre"], "propiedades": props}
        )




    # Almacenamiento en formato JSON
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)

    print(f"\nProceso finalizado. Datos guardados en {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Scrapeando Montevideo...


/var/folders/hg/2mj0mwts6rs11fbd4lnwkvn00000gn/T/ipykernel_33382/3335775556.py:74: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  size_tag = card.find(text=re.compile(r"m²|m2", re.I))
/var/folders/hg/2mj0mwts6rs11fbd4lnwkvn00000gn/T/ipykernel_33382/3335775556.py:80: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  beds_tag = card.find(text=re.compile(r"dorm|hab|dormitorio", re.I))


✓ Montevideo: 10 propiedades extraídas correctamente.
Scrapeando Punta del Este...
✓ Punta del Este: 10 propiedades extraídas correctamente.
Scrapeando Ciudad de la Costa...
✓ Ciudad de la Costa: 10 propiedades extraídas correctamente.

Proceso finalizado. Datos guardados en propiedades.json
